Baseline

In [2]:
import pandas as pd
df = pd.read_csv("../data/raw/creditcard.csv")

In [3]:
df = df.sample(n=50000, random_state=42)

In [4]:
# 分层切分数据，保持训练/测试集的欺诈比例一致
from sklearn.model_selection import train_test_split
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 用 Pipeline 把标准化和逻辑回归串起来，避免数据泄漏
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
pipe.fit(X_train, y_train)

# 输出关键指标——因为样本极度不平衡，重点看 PR-AUC 和 Recall
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
y_proba = pipe.predict_proba(X_test)[:, 1]
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC :", average_precision_score(y_test, y_proba))
print(classification_report(y_test, y_proba > 0.5))

ROC-AUC: 0.9282103104689737
PR-AUC : 0.6698412634215712
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      9983
           1       0.05      0.76      0.10        17

    accuracy                           0.98     10000
   macro avg       0.53      0.87      0.54     10000
weighted avg       1.00      0.98      0.99     10000

